In [ ]:
import ee
import geemap
import time
import os
from google.colab import files
import geopandas as gpd
import json
import numpy as np
from glob import glob
from google.colab import drive
import matplotlib.pyplot as plt

# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='Here Add your project name from GEE account')    # eg. project name: 'gee-tutorial-470209'

# Add Study area as geojason. Projection must be in wgs84

In [ ]:
# Define the desired directory path
target_directory = '/content/Aoi'
uploaded = files.upload(target_directory)

Saving AOI_211.geojson to /content/Aoi/AOI_211.geojson


In [ ]:
# Load GeoJSON
with open("/content/Aoi/AOI_211.geojson") as f:
    geojson_data = json.load(f)

# Convert to EE Geometry or FeatureCollection
roi = ee.FeatureCollection(geojson_data)
first_feature = roi.first()
print('Segment Processing Start',first_feature.get('FID').getInfo())

Segment Processing Start 17


# NIR R G Collection 2026 to 1988 Descending order

This script implements a multi-sensor remote sensing methodology in the Google Earth Engine Python API to generate annual dry-season false-color median composite imagery (1988–2026) over a user-defined ROI. The workflow collects imagery from Landsat program (Landsat 5, 7, and 8) and Copernicus Programme Sentinel-2, applies sensor-specific radiometric scaling, filters scenes by cloud thresholds, and uses median compositing during the December–March low-flow period to reduce cloud contamination and seasonal variability. A hierarchical sensor-priority approach favors higher-resolution imagery when available, producing consistent annual NIR–Red–Green false-color composites suitable for long-term channel and river morphology analysis.


In [ ]:
from tqdm import tqdm
import time
import ee

# =================================================
# SCALING FUNCTIONS
# =================================================

def scale_landsat5(img):
    sr = img.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    scaled = sr.multiply(0.0000275).add(-0.2)
    return img.addBands(scaled, overwrite=True)

def scale_landsat7(img):
    sr = img.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    scaled = sr.multiply(0.0000275).add(-0.2)
    return img.addBands(scaled, overwrite=True)

def scale_landsat8(img):
    sr = img.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'])
    scaled = sr.multiply(0.0000275).add(-0.2)
    return img.addBands(scaled, overwrite=True)

def scale_sentinel2(img):
    sr = img.select(['B2','B3','B4','B8'])  # Blue, Green, Red, NIR
    scaled = sr.divide(10000)
    return img.addBands(scaled, overwrite=True)

# =================================================
# FALSE COLOR COLLECTION FUNCTION
# =================================================

def get_false_color_image(year, debug=False):
    start = f"{year-1}-12-01"
    end   = f"{year}-04-01"

    if debug:
        print(f"  {year}: Searching {start} to {end}")

    satellites = [
        {
            'name': 'S2',
            'display': 'Sentinel-2',
            'collection': 'COPERNICUS/S2_SR_HARMONIZED',
            'scale_func': scale_sentinel2,
            'bands': {'nir': 'B8', 'red': 'B4', 'green': 'B3'},
            'cloud_prop': 'CLOUDY_PIXEL_PERCENTAGE',
            'cloud_cover': 40,   # relaxed for Bangladesh
            'resolution': 10
        },
        {
            'name': 'LS8',
            'display': 'Landsat 8',
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'scale_func': scale_landsat8,
            'bands': {'nir': 'SR_B5', 'red': 'SR_B4', 'green': 'SR_B3'},
            'cloud_prop': 'CLOUD_COVER',
            'cloud_cover': 30,
            'resolution': 30
        },
        {
            'name': 'LS5',
            'display': 'Landsat 5',
            'collection': 'LANDSAT/LT05/C02/T1_L2',
            'scale_func': scale_landsat5,
            'bands': {'nir': 'SR_B4', 'red': 'SR_B3', 'green': 'SR_B2'},
            'cloud_prop': 'CLOUD_COVER',
            'cloud_cover': 20,
            'resolution': 30
        },
        {
            'name': 'LS7',
            'display': 'Landsat 7',
            'collection': 'LANDSAT/LE07/C02/T1_L2',
            'scale_func': scale_landsat7,
            'bands': {'nir': 'SR_B4', 'red': 'SR_B3', 'green': 'SR_B2'},
            'cloud_prop': 'CLOUD_COVER',
            'cloud_cover': 20,
            'resolution': 30
        }
    ]

    for sat in satellites:
        try:
            col = (ee.ImageCollection(sat['collection'])
                   .filterBounds(roi)
                   .filterDate(start, end)
                   .map(sat['scale_func']))

            # ✅ Correct cloud filtering
            col = col.filter(ee.Filter.lt(sat['cloud_prop'], sat['cloud_cover']))

            count = col.size().getInfo()

            if debug and sat['name'] == 'S2' and count == 0:
                col_all = (ee.ImageCollection(sat['collection'])
                           .filterBounds(roi)
                           .filterDate(start, end))
                total = col_all.size().getInfo()
                print(f"  Debug {year}: S2 total={total}, after cloud filter=0")

            if count > 0:
                median = col.median().clip(roi)

                false_color = median.select([
                    sat['bands']['nir'],
                    sat['bands']['red'],
                    sat['bands']['green']
                ]).rename(['NIR', 'Red', 'Green'])

                false_color = false_color.set({
                    'source': sat['name'],
                    'source_display': sat['display'],
                    'year': year,
                    'start_date': start,
                    'end_date': end,
                    'n_images': count,
                    'resolution': sat['resolution']
                })

                if sat['name'] == 'LS7':
                    false_color = false_color.set({
                        'note': 'SLC-off stripes possible after 2003'
                    })

                print(f"  ✓ {year}: {sat['display']} ({count} images, {sat['resolution']}m)")
                return false_color

        except Exception as e:
            if debug:
                print(f"  ✗ {year}: {sat['display']} failed - {str(e)}")

    print(f"  ✗ {year}: No data available")
    return None


# =================================================
# MAIN COLLECTION FUNCTION
# =================================================

def collect_all_years(roi_geometry, start_year=1988, end_year=2026, debug=False):

    global roi
    roi = roi_geometry

    years = list(range(end_year, start_year - 1, -1))
    all_images = []

    print("\n" + "=" * 80)
    print("COLLECTING FALSE COLOR IMAGES")
    print("=" * 80)

    for year in tqdm(years):
        try:
            img = get_false_color_image(year, debug)
            if img:
                all_images.append(img)
            time.sleep(0.3)
        except Exception as e:
            print(f"Error {year}: {str(e)}")

    print("\n" + "=" * 80)
    print(f"Collected {len(all_images)}/{len(years)} years")

    if all_images:
        collection = ee.ImageCollection(all_images)

        # Summary
        sources = {}
        for img in all_images:
            src = img.get('source_display').getInfo()
            sources[src] = sources.get(src, 0) + 1

        print("\n📊 SUMMARY:")
        for k, v in sources.items():
            print(f"{k}: {v} years")

        return collection, all_images

    return None, []


# =================================================
# RUN
# =================================================

collection, images_list = collect_all_years(
    roi_geometry=roi,
    start_year=1988,
    end_year=2026,
    debug=True
)

# Print results
for img in images_list:
    year = img.get('year').getInfo()
    source = img.get('source_display').getInfo()
    n_images = img.get('n_images').getInfo()
    print(f"{year}: {source} ({n_images} images)")


COLLECTING FALSE COLOR IMAGES


  0%|          | 0/39 [00:00<?, ?it/s]

  2026: Searching 2025-12-01 to 2026-04-01
  ✓ 2026: Sentinel-2 (76 images, 10m)


  3%|▎         | 1/39 [00:00<00:29,  1.27it/s]

  2025: Searching 2024-12-01 to 2025-04-01
  ✓ 2025: Sentinel-2 (88 images, 10m)


...

  1989: Searching 1988-12-01 to 1989-04-01
  Debug 1989: S2 total=0, after cloud filter=0
  ✓ 1989: Landsat 5 (17 images, 30m)


 97%|█████████▋| 38/39 [00:43<00:01,  1.19s/it]

  1988: Searching 1987-12-01 to 1988-04-01
  Debug 1988: S2 total=0, after cloud filter=0
  ✓ 1988: Landsat 5 (10 images, 30m)


100%|██████████| 39/39 [00:44<00:00,  1.14s/it]



Collected 39/39 years

📊 SUMMARY:
Sentinel-2: 11 years
Landsat 8: 2 years
Landsat 7: 4 years
Landsat 5: 22 years
2026: Sentinel-2 (76 images)
2025: Sentinel-2 (88 images)
2024: Sentinel-2 (54 images)
2023: Sentinel-2 (76 images)
2022: Sentinel-2 (63 images)
2021: Sentinel-2 (72 images)
2020: Sentinel-2 (56 images)
2019: Sentinel-2 (73 images)
2018: Sentinel-2 (18 images)
2017: Sentinel-2 (14 images)
2016: Sentinel-2 (6 images)
2015: Landsat 8 (12 images)
2014: Landsat 8 (16 images)
2013: Landsat 7 (17 images)
2012: Landsat 7 (12 images)
2011: Landsat 5 (15 images)
2010: Landsat 5 (7 images)
2009: Landsat 5 (12 images)
2008: Landsat 5 (4 images)
2007: Landsat 5 (15 images)
2006: Landsat 5 (7 images)
2005: Landsat 5 (10 images)
2004: Landsat 5 (12 images)
2003: Landsat 7 (16 images)
2002: Landsat 7 (10 images)
2001: Landsat 5 (13 images)
2000: Landsat 5 (14 images)
1999: Landsat 5 (15 images)
1998: Landsat 5 (13 images)
1997: Landsat 5 (10 images)
1996: Landsat 5 (4 images)
1995: Landsa

# Add to map for visualization

In [ ]:
import geemap
import ipywidgets as widgets
from IPython.display import display

# Create map
Map = geemap.Map()
Map.centerObject(roi, 8)

# Convert list to dictionary {year: image}
image_dict = {}
for img in images_list:
    year = img.get('year').getInfo()
    image_dict[year] = img

# Sort years
years_sorted = sorted(image_dict.keys())

# Visualization parameters (False Color: NIR, Red, Green)
vis_params = {
    'min': 0,
    'max': 0.4,
    'bands': ['NIR', 'Red', 'Green']
}

# Dropdown widget
dropdown = widgets.Dropdown(
    options=years_sorted,
    description='Year:',
    value=years_sorted[0],
    layout=widgets.Layout(width='200px')
)

# Function to update map
def update_map(change):
    year = change['new']

    Map.layers = Map.layers[:1]  # keep base layer only

    img = image_dict[year]
    source = img.get('source_display').getInfo()

    Map.addLayer(img, vis_params, f"{year} - {source}")
    print(f"Showing: {year} ({source})")

# Attach event
dropdown.observe(update_map, names='value')

# Initial display
Map.addLayer(image_dict[years_sorted[0]], vis_params, f"{years_sorted[0]}")

# Display UI
display(dropdown)
Map

Dropdown(description='Year:', layout=Layout(width='200px'), options=(1988, 1989, 1990, 1991, 1992, 1993, 1994,…

Map(center=[23.927511976252976, 89.94871451074872], controls=(WidgetControl(options=['position', 'transparent_…

Showing: 1996 (Landsat 5)
Showing: 2003 (Landsat 7)
Showing: 2024 (Sentinel-2)


# Export to Drive

In [ ]:
def export_all_to_drive_batch(images_list, folder_name="False_Color_Image_Padma",
                              scale_override=None, delay=10, batch_size=5):
    """
    Export images in batches to avoid overwhelming Earth Engine
    """

    print("\n" + "=" * 80)
    print("STARTING BATCH EXPORT")
    print("=" * 80)
    print(f"Total images: {len(images_list)}")
    print(f"Batch size: {batch_size}")
    print(f"Delay between batches: {delay * 2} seconds")
    print("=" * 80)

    export_region = roi.geometry().bounds()
    successful = 0
    failed = 0

    for batch_start in range(0, len(images_list), batch_size):
        batch_end = min(batch_start + batch_size, len(images_list))
        print(f"\n📦 Processing batch {batch_start//batch_size + 1}: images {batch_start+1} to {batch_end}")
        print("-" * 40)

        for i in range(batch_start, batch_end):
            img = images_list[i]
            try:
                year = img.get('year').getInfo()
                source = img.get('source').getInfo()
                resolution = img.get('resolution').getInfo()

                file_name = f"{year}_{source}_NIR_R_G_Image"
                scale = scale_override if scale_override else resolution
                clean_img = img.select(['NIR', 'Red', 'Green'])

                task = ee.batch.Export.image.toDrive(
                    image=clean_img,
                    description=file_name,
                    folder=folder_name,
                    fileNamePrefix=file_name,
                    region=export_region,
                    scale=scale,
                    maxPixels=1e13,
                    fileFormat='GeoTIFF'
                )

                task.start()
                successful += 1
                print(f"  ✔ [{i+1}/{len(images_list)}] {file_name}")

                if i < batch_end - 1:  # Delay between exports in batch
                    time.sleep(delay)

            except Exception as e:
                failed += 1
                print(f"  ✗ [{i+1}/{len(images_list)}] Failed: {str(e)[:100]}")

        # Longer delay between batches
        if batch_end < len(images_list):
            print(f"  ⏳ Waiting {delay*2}s before next batch...")
            time.sleep(delay * 2)

    print("\n" + "=" * 80)
    print("EXPORT COMPLETE")
    print("=" * 80)
    print(f"✓ Successful: {successful}/{len(images_list)}")
    print(f"✗ Failed: {failed}")
    print(f"📁 Check your Google Drive folder: {folder_name}")
    print("\n💡 Tip: Check export status in Earth Engine Code Editor:")
    print("   https://code.earthengine.google.com/tasks")
    print("=" * 80)

# Use it:
export_all_to_drive_batch(images_list, folder_name="False_Color_Image_Padma",
                          scale_override=None, delay=10, batch_size=5)


STARTING BATCH EXPORT
Total images: 39
Batch size: 5
Delay between batches: 20 seconds

📦 Processing batch 1: images 1 to 5
----------------------------------------
  ✔ [1/39] 2026_S2_NIR_R_G_Image
  ✔ [2/39] 2025_S2_NIR_R_G_Image
  ✔ [3/39] 2024_S2_NIR_R_G_Image
  ✔ [4/39] 2023_S2_NIR_R_G_Image
  ✔ [5/39] 2022_S2_NIR_R_G_Image
  ⏳ Waiting 20s before next batch...

📦 Processing batch 2: images 6 to 10
----------------------------------------
  ✔ [6/39] 2021_S2_NIR_R_G_Image
  ✔ [7/39] 2020_S2_NIR_R_G_Image
  ✔ [8/39] 2019_S2_NIR_R_G_Image
  ✔ [9/39] 2018_S2_NIR_R_G_Image
  ✔ [10/39] 2017_S2_NIR_R_G_Image
  ⏳ Waiting 20s before next batch...

📦 Processing batch 3: images 11 to 15
----------------------------------------
  ✔ [11/39] 2016_S2_NIR_R_G_Image
  ✔ [12/39] 2015_LS8_NIR_R_G_Image
  ✔ [13/39] 2014_LS8_NIR_R_G_Image
  ✔ [14/39] 2013_LS7_NIR_R_G_Image
  ✔ [15/39] 2012_LS7_NIR_R_G_Image
  ⏳ Waiting 20s before next batch...

📦 Processing batch 4: images 16 to 20
--------------------

In [ ]:
tasks = ee.batch.Task.list()
for t in tasks:
    print(t.config.get('description'), "→", t.status()['state'])

1988_LS5_NIR_R_G_Image → READY
1989_LS5_NIR_R_G_Image → READY
1990_LS5_NIR_R_G_Image → READY
1991_LS5_NIR_R_G_Image → READY
1992_LS5_NIR_R_G_Image → READY
1993_LS5_NIR_R_G_Image → READY
1994_LS5_NIR_R_G_Image → READY
1995_LS5_NIR_R_G_Image → READY
1996_LS5_NIR_R_G_Image → READY
1997_LS5_NIR_R_G_Image → READY
1998_LS5_NIR_R_G_Image → READY
1999_LS5_NIR_R_G_Image → READY
2000_LS5_NIR_R_G_Image → READY
2001_LS5_NIR_R_G_Image → READY
2002_LS7_NIR_R_G_Image → READY
2003_LS7_NIR_R_G_Image → READY
2004_LS5_NIR_R_G_Image → READY
2005_LS5_NIR_R_G_Image → READY
2006_LS5_NIR_R_G_Image → READY
2007_LS5_NIR_R_G_Image → READY
2008_LS5_NIR_R_G_Image → READY
2009_LS5_NIR_R_G_Image → READY
2010_LS5_NIR_R_G_Image → READY
2011_LS5_NIR_R_G_Image → READY
2012_LS7_NIR_R_G_Image → READY
2013_LS7_NIR_R_G_Image → READY
2014_LS8_NIR_R_G_Image → READY
2015_LS8_NIR_R_G_Image → READY
2016_S2_NIR_R_G_Image → READY
2017_S2_NIR_R_G_Image → READY
2018_S2_NIR_R_G_Image → READY
2019_S2_NIR_R_G_Image → READY
2020_S2_NIR_